# LLM 9일차 (2) — 멀티 에이전트 RAG 챗봇을 비동기로 : `async def` 노드 · `await ainvoke` · `main()`

수업 필기 `LangGraph_실습6` 을 정리 · 완성한 노트입니다. 필기는 비동기 노드 3개까지 만들고 **그래프 구성이 비어 있고 `main()` 이 미완성**이라, 교재 p.125~128 의 '비동기 처리 방식으로의 변화 3단계' 대로 채웠습니다.
(교재 `AI Agent` **p.125~128** — 1단계 `def` → `async def` · 2단계 `invoke` → `await … ainvoke` · 3단계 `main()` + `asyncio.run` / 실습 : 멀티 에이전트 RAG 시스템 업그레이드(1))

| 단계 | 동기 (`LangGraph_실습5`) | 비동기 (`LangGraph_실습6`) |
|---|---|---|
| 1 노드 | `def rephrase_node(state)` | `async def rephrase_node(state)` |
| 2 I/O 호출 | `rephrase_chain.invoke(...)` · `retriever.invoke(...)` | `await rephrase_chain.ainvoke(...)` · `await retriever.ainvoke(...)` |
| 3 실행 | `multi_agent.invoke(inputs)` | `async def main(): … await multi_agent.ainvoke(inputs)` → `await main()` (Jupyter) / `asyncio.run(main())` (.py) |

- 비동기 기초(코루틴 · `gather` · 막는 호출)는 [`python_asyncio_basics.ipynb`](./python_asyncio_basics.ipynb) · 동기판 수업 실행 기록은 [`langgraph_multi_agent_class_run.ipynb`](./langgraph_multi_agent_class_run.ipynb)
- 원본에서 고친 부분은 **📝 고친 부분** 으로 적었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

> **출력 읽는 법** — 정리 노트북은 **API 키 없이** 만들었다.
>
> | 표시 | 뜻 |
> |---|---|
> | `# ⏳` | OpenAI 키가 필요해 **출력이 비어 있는 셀** — 직접 실행 |
> | `# 🧪` | OpenAI 대신 **로컬 임베딩** · **0.3초 기다리는 가짜 LLM** 으로 실제로 돌린 셀 — 답변 문장과 시간은 gpt-5-nano 와 다르다. **비동기 동작(오류 · 실행 순서 · 동시 실행 시간 단축)은 진짜와 같다** |
> | 표시 없음 | 정리 노트북에서 실제로 실행한 셀 |

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `langgraph` | `StateGraph` · `ainvoke` | `pip install langgraph` |
| `langchain-openai` · `langchain-community` · `langchain-chroma` | `ChatOpenAI` · `OpenAIEmbeddings` · `CSVLoader` · `Chroma` | `pip install langchain-openai langchain-community langchain-chroma` |
| `langgraph-checkpoint-sqlite` | 필기 첫 셀의 `AsyncSqliteSaver` (다음 시간 체크포인팅 · 오늘은 쓰지 않음) | `pip install langgraph-checkpoint-sqlite` |
| `langchain-huggingface` | (🧪 키 없이) 로컬 임베딩 | `pip install langchain-huggingface sentence-transformers` |

1. 같은 폴더에 `faq_manual.csv`(Day 053 폴더 · 이 폴더에도 있음) · `.env`(`OPENAI_API_KEY`) — **`.env` 는 올리지 않는다**
2. 키가 있으면 `# ⏳` 셀, 없으면 `# 🧪` 셀 (정리 환경 : `langgraph 1.2.21` · `langchain-core 1.8.0` · `langgraph-checkpoint-sqlite 3.1.3`)

> **📝 고친 부분** 필기 첫 셀의 `from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver` 는 `langgraph-checkpoint-sqlite` 를 따로 설치하지 않으면 `ModuleNotFoundError` 다(정리 환경에서 확인). 오늘 코드에서는 쓰지 않으므로 다음 시간(체크포인팅, 교재 p.129~)으로 미뤘다.

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q langgraph langchain-openai langchain-community langchain-chroma python-dotenv

In [2]:
import os, time, asyncio, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
from typing import Annotated
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel
from langchain_community.document_loaders import CSVLoader
from langchain_chroma import Chroma
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END, add_messages

loaded = load_dotenv(find_dotenv(usecwd=True), override=True)       # 키 값은 출력하지 않는다
print('.env 읽음 :', loaded, '| OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')))

.env 읽음 : False | OPENAI_API_KEY 있음 : False


## 1. RAG 기반 설정 (필기와 같음 · Day 053)

In [3]:
docs = CSVLoader('faq_manual.csv', encoding='utf-8',
                 content_columns=['question', 'answer'], metadata_columns=['faq_id', 'category']).load()
print('문서 수 :', len(docs))

문서 수 : 10


In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
model = OpenAIEmbeddings(model='text-embedding-3-small')
llm = ChatOpenAI(model='gpt-5-nano')

In [5]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 거리는 text-embedding-3-small 과 다를 수 있다

from langchain_huggingface import HuggingFaceEmbeddings
model = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask', encode_kwargs={'normalize_embeddings': True})

In [6]:
# 🧪 LLM 대신 '0.3초 기다렸다가 정해 둔 규칙으로 답하는' 가짜 LLM — API 응답 대기(I/O)를 흉내 내 시간을 잰다

from langchain_core.runnables import RunnableLambda

LATENCY = 0.3          # API 응답을 기다리는 시간 흉내

def _answer(prompt_value):
    msgs = prompt_value.to_messages()
    system, last, history = msgs[0].content, msgs[-1].content, msgs[1:-1]
    if '재구성' in system:                              # 질문 재구성 : 앞 사용자 질문 + 지금 질문 (Day 053 과 같은 규칙)
        prev = [m.content for m in history if isinstance(m, HumanMessage)]
        return AIMessage(f'{prev[-1]} {last}' if prev else last)
    context = system.split('[Context]')[-1]
    first = context.split('answer: ')[-1].split('. ')[0].strip() if 'answer: ' in context else '(없음)'
    return AIMessage(f'(모의 답변 · 기록 {len(history)}개) {first}.')

def _sync_llm(pv):
    time.sleep(LATENCY); return _answer(pv)             # 동기 invoke : 멈춰서 기다림

async def _async_llm(pv):
    await asyncio.sleep(LATENCY); return _answer(pv)    # 비동기 ainvoke : 기다리는 동안 양보

llm = RunnableLambda(_sync_llm, afunc=_async_llm)       # invoke / ainvoke 를 모두 가진 가짜 LLM

In [7]:
vectorstore = Chroma.from_documents(docs, model, collection_name='faq_async')    # 메모리 저장소
retriever = vectorstore.as_retriever(search_kwargs={'k': 1})
print('환불은 →', (await retriever.ainvoke('환불은'))[0].metadata['faq_id'])   # 검색기도 ainvoke 가 있다

환불은 → FAQ-002


## 2. 전용 체인 2개 (필기)

In [8]:
rephrase_system_prompt = '''
주어진 채팅 기록(chat history)과 최신 사용자 질문이 있을 때,
이 질문이 채팅 기록의 맥락을 참조하고 있다면 모호한 질문(예: "그거 전액 다 받으려면 조건이 뭐야?")을,
채팅 기록 없이도 이해할 수 있는 명확한 질문(예: "수강 신청한 강의를 전액 환불 받으려면 조건이 뭐야?")으로 다시 작성하세요.
질문에 답하지 말고, 오직 질문을 재구성하거나 그대로 반환하기만 하세요.
'''
rephrase_prompt = ChatPromptTemplate.from_messages([
    ('system', rephrase_system_prompt), MessagesPlaceholder('chat_history'), ('human', '{input}')])
rephrase_chain = rephrase_prompt | llm | StrOutputParser()

qa_system_prompt = '''
당신은 코드크래프터스 아카데미의 친절한 고객 지원 담당자입니다.
아래의 [Context] 정보를 사용하여 사용자의 질문에 답변하세요.
정보가 없다면 '죄송하지만 매뉴얼에서 정보를 찾을 수 없습니다'라고 답하세요.

[Context]
{context}
'''
qa_prompt = ChatPromptTemplate.from_messages([
    ('system', qa_system_prompt), MessagesPlaceholder('chat_history'), ('human', '{input}')])
qa_chain = qa_prompt | llm                        # 결과가 AIMessage 그대로 → 노드가 messages 에 바로 넣는다

## 3. 상태 · 비동기 노드 3개 (교재 p.125~127 · 필기)

In [9]:
class AgentState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = []
    rephrased_query: str = ''
    context: str = ''

async def rephrase_node(state: AgentState) -> dict:           # 1단계 : def → async def
    latest_question = state.messages[-1].content
    chat_history = state.messages[:-1]
    if chat_history:                                          # 앞 대화가 있을 때만 LLM 호출 (첫 질문은 그대로)
        query = await rephrase_chain.ainvoke({'chat_history': chat_history, 'input': latest_question})   # 2단계
    else:
        query = latest_question
    print(f'  [에이전트 1] 재구성 : {query}')
    return {'rephrased_query': query}

async def retriever_node(state: AgentState) -> dict:
    retrieved_docs = await retriever.ainvoke(state.rephrased_query)
    print(f"  [에이전트 2] 검색 : {[d.metadata['faq_id'] for d in retrieved_docs]}")
    return {'context': '\n\n'.join(d.page_content for d in retrieved_docs)}

async def answer_node(state: AgentState) -> dict:
    response = await qa_chain.ainvoke({'context': state.context, 'chat_history': state.messages[:-1],
                                       'input': state.messages[-1].content})
    print(f'  [에이전트 3] 답변 : {response.content[:60]}')
    return {'messages': [response]}

## 4. 그래프 구성 · 컴파일 (필기에서 비어 있던 곳)

> **📝 고친 부분** 필기는 주석 `# 비동기 처리 방식 노드 함수 -> 그래프 -> 노드 -> compile -> mult_agent` 만 있고 코드가 없다(`mult_agent` 도 오타). 그래프 만드는 법은 동기판(`LangGraph_실습5`)과 **완전히 같다** — 노드가 `async def` 라도 `add_node` · `add_edge` · `compile` 은 그대로다.

In [10]:
graph = StateGraph(AgentState)
graph.add_node('rephrase_node', rephrase_node)
graph.add_node('retriever_node', retriever_node)
graph.add_node('answer_node', answer_node)
graph.set_entry_point('rephrase_node')
graph.add_edge('rephrase_node', 'retriever_node')
graph.add_edge('retriever_node', 'answer_node')
graph.add_edge('answer_node', END)
multi_agent = graph.compile()

In [11]:
# 비동기 노드가 든 그래프를 동기 invoke 로 부르면?
try:
    multi_agent.invoke({'messages': [HumanMessage('환불은 어떻게?')]})
except TypeError as e:
    print('TypeError :', str(e).splitlines()[0])
    print('          ', str(e).splitlines()[1])

TypeError : No synchronous function provided to "rephrase_node".
           Either initialize with a synchronous function or invoke via the async API (ainvoke, astream, etc.)


- `async def` 노드는 **`ainvoke` · `astream` 으로만** 실행할 수 있다. 반대로 동기 `def` 노드는 `ainvoke` 로도 돌릴 수 있다(LangGraph 가 스레드에서 실행 — §6 에서 확인)

## 5. 실행 — `main()` (교재 p.128 3단계)

### 필기 `main()` 그대로 실행하면

In [12]:
# 필기 : final_state = await multi_agent.ainvoke()   ← 입력이 없다
try:
    await multi_agent.ainvoke()
except TypeError as e:
    print('TypeError :', e)

TypeError : Pregel.ainvoke() missing 1 required positional argument: 'input'


> **📝 고친 부분** 필기 `main()` 은 ① `ainvoke()` 에 **입력(inputs)을 주지 않아** 매번 `TypeError` 가 나고 ② 그 오류를 `except` 가 잡아 출력만 한 뒤 `while True` 로 돌아가며 ③ `input()` · `exit` 처리가 없어서, `await main()` 을 실행하면 **같은 오류 문장을 끝없이 출력하는 무한 반복**이 된다(위 오류가 매 바퀴 반복). 동기판 `실습5` 의 루프(입력 받기 → `exit` 확인 → `inputs` 에 질문 추가 → 실행 → `inputs = final_state`)를 비동기로 옮기고, 오류가 나면 반복을 멈추도록 고쳤다.

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

# 고친 main() — 직접 입력하며 대화 ('exit' 로 종료)
async def main():
    print("안녕하세요! 코드크래프터스 아카데미 지원 챗봇입니다. 무엇이든 물어보세요! (종료: exit)")
    inputs = {'messages': []}
    while True:
        user_input = input('사용자: ')
        if user_input.strip().lower() == 'exit':
            print('챗봇을 안전하게 종료합니다.'); break
        inputs['messages'].append(HumanMessage(content=user_input))
        try:
            final_state = await multi_agent.ainvoke(inputs)        # 3단계 : await … ainvoke
        except Exception as e:
            print(f'오류가 발생했습니다: {e}'); break             # 📝 같은 오류를 끝없이 반복하지 않도록
        print(f"AI 에이전트:\n{final_state['messages'][-1].content}\n" + '-' * 80)
        inputs = final_state                                        # 다음 턴에 대화 기록을 이어 준다

await main()                      # Jupyter · VS Code 노트북
# asyncio.run(main())             # .py 파일

In [14]:
# 🧪 LLM 대신 '0.3초 기다렸다가 정해 둔 규칙으로 답하는' 가짜 LLM — API 응답 대기(I/O)를 흉내 내 시간을 잰다
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 거리는 text-embedding-3-small 과 다를 수 있다

# 같은 main() 을 '입력 목록' 으로 흉내 — input() 대신 정해 둔 질문을 차례로
async def main(questions):
    inputs = {'messages': []}
    for user_input in questions:
        print(f'사용자: {user_input}')
        if user_input.strip().lower() == 'exit':
            print('챗봇을 안전하게 종료합니다.'); break
        inputs['messages'].append(HumanMessage(content=user_input))
        final_state = await multi_agent.ainvoke(inputs)
        print(f"  → messages {len(final_state['messages'])}개\n")
        inputs = final_state

await main(['수강 신청한 강의를 환불하고 싶어요', '그거 전액 다 받으려면 조건이 어떻게 되나요?', 'exit'])

사용자: 수강 신청한 강의를 환불하고 싶어요
  [에이전트 1] 재구성 : 수강 신청한 강의를 환불하고 싶어요
  [에이전트 2] 검색 : ['FAQ-002']


  [에이전트 3] 답변 : (모의 답변 · 기록 0개) 네, 환불 가능합니다.
  → messages 2개

사용자: 그거 전액 다 받으려면 조건이 어떻게 되나요?


  [에이전트 1] 재구성 : 수강 신청한 강의를 환불하고 싶어요 그거 전액 다 받으려면 조건이 어떻게 되나요?
  [에이전트 2] 검색 : ['FAQ-002']


  [에이전트 3] 답변 : (모의 답변 · 기록 2개) 네, 환불 가능합니다.
  → messages 4개

사용자: exit
챗봇을 안전하게 종료합니다.


- 첫 질문은 기록이 없어 재구성 LLM 을 **부르지 않고** 그대로 검색한다(필기의 `if chat_history:`) — 호출 1번을 아낀다
- 두 번째 질문은 앞 대화가 이어져(messages 3개 → 4개) 환불 FAQ(FAQ-002)를 찾았다 — 동기판 수업 실행(📼 `langgraph_multi_agent_class_run`)과 같은 흐름
- `input()` 은 사용자가 칠 때까지 **이벤트 루프 전체를 멈추는** 동기 함수다. 혼자 쓰는 콘솔 챗봇은 괜찮지만, 여러 사용자를 동시에 받는 서버라면 웹 프레임워크(FastAPI 등)가 입력을 받아 `ainvoke` 를 부르게 한다

## 6. 그래서 빨라지나? — 동기 vs 비동기 시간 재기

같은 일을 하는 **동기 노드 그래프**를 하나 더 만들어 비교했다. 가짜 LLM 은 호출마다 0.3초 기다린다(두 번째 질문은 재구성 + 답변 = LLM 2번).

In [15]:
# 🧪 LLM 대신 '0.3초 기다렸다가 정해 둔 규칙으로 답하는' 가짜 LLM — API 응답 대기(I/O)를 흉내 내 시간을 잰다

def rephrase_sync(state: AgentState) -> dict:
    h = state.messages[:-1]
    q = rephrase_chain.invoke({'chat_history': h, 'input': state.messages[-1].content}) if h else state.messages[-1].content
    return {'rephrased_query': q}
def retriever_sync(state: AgentState) -> dict:
    return {'context': '\n\n'.join(d.page_content for d in retriever.invoke(state.rephrased_query))}
def answer_sync(state: AgentState) -> dict:
    return {'messages': [qa_chain.invoke({'context': state.context, 'chat_history': state.messages[:-1],
                                          'input': state.messages[-1].content})]}

g = StateGraph(AgentState)
for name, fn in [('rephrase_node', rephrase_sync), ('retriever_node', retriever_sync), ('answer_node', answer_sync)]:
    g.add_node(name, fn)
g.set_entry_point('rephrase_node'); g.add_edge('rephrase_node', 'retriever_node')
g.add_edge('retriever_node', 'answer_node'); g.add_edge('answer_node', END)
sync_agent = g.compile()

conversation = [HumanMessage('수강 신청한 강의를 환불하고 싶어요'), AIMessage('7일 이내 · 5% 미만이면 전액 환불됩니다.'),
                HumanMessage('그거 전액 다 받으려면 조건이 어떻게 되나요?')]
users = [{'messages': list(conversation)} for _ in range(5)]          # 같은 질문을 보낸 사용자 5명

import contextlib, io
quiet = lambda: contextlib.redirect_stdout(io.StringIO())             # 노드의 print 를 잠시 끔

with quiet():
    t = time.perf_counter(); sync_agent.invoke(users[0]); one_sync = time.perf_counter() - t
    t = time.perf_counter(); await multi_agent.ainvoke(users[0]); one_async = time.perf_counter() - t
    t = time.perf_counter(); [sync_agent.invoke(u) for u in users]; five_sync = time.perf_counter() - t
    t = time.perf_counter(); [await multi_agent.ainvoke(u) for u in users]; five_seq = time.perf_counter() - t
    t = time.perf_counter(); await asyncio.gather(*(multi_agent.ainvoke(u) for u in users)); five_gather = time.perf_counter() - t

print(f'사용자 1명 : 동기 invoke {one_sync:.2f}초 · 비동기 ainvoke {one_async:.2f}초')
print(f'사용자 5명 : 동기 차례로 {five_sync:.2f}초 · 비동기 차례로 await {five_seq:.2f}초 · 비동기 gather {five_gather:.2f}초')

사용자 1명 : 동기 invoke 0.67초 · 비동기 ainvoke 0.67초
사용자 5명 : 동기 차례로 3.35초 · 비동기 차례로 await 3.35초 · 비동기 gather 0.94초


- **한 사람의 대화는 비동기로 바꿔도 빨라지지 않는다** — 재구성 → 검색 → 답변은 앞 결과가 있어야 다음을 하므로 차례로 기다릴 수밖에 없다
- 이득은 **여러 요청이 동시에 들어올 때**다. 5명을 `gather` 로 동시에 처리하면 기다림이 겹쳐 '차례로' 의 1/3 쯤(위 출력 : 약 3.3초 → 약 0.9초)에 끝난다. 웹 서비스 · 여러 질문 일괄 처리 · 병렬 노드(다음 단원)에서 쓰인다
- 📝 교재 p.120~122 '비동기 = 효율' 은 이런 **동시성**의 이야기다. 오늘 실습6 처럼 콘솔에서 한 명이 묻는 챗봇은 비동기로 바꿔도 체감 속도는 같다

In [16]:
# 🧪 LLM 대신 '0.3초 기다렸다가 정해 둔 규칙으로 답하는' 가짜 LLM — API 응답 대기(I/O)를 흉내 내 시간을 잰다

# 동기 def 노드 그래프도 ainvoke + gather 로 돌리면? (LangGraph 가 동기 노드를 스레드에서 실행)
with quiet():
    t = time.perf_counter(); await asyncio.gather(*(sync_agent.ainvoke(u) for u in users)); sync_nodes_gather = time.perf_counter() - t
print(f'동기 노드 그래프 · ainvoke + gather (5명) : {sync_nodes_gather:.2f}초')

동기 노드 그래프 · ainvoke + gather (5명) : 0.95초


- 동기 노드도 `ainvoke` 로 부르면 LangGraph 가 **스레드**에 넘겨 겹쳐 실행한다(위 시간). 그래도 스레드는 수가 제한되고 무겁기 때문에, 수백 개 요청을 받을 서비스라면 노드를 `async def` + `ainvoke` 로 쓰는 것이 교재가 말하는 방향이다

## 7. ⏳ 진짜 gpt-5-nano 로 시간 재기

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

llm = ChatOpenAI(model='gpt-5-nano')
rephrase_chain = rephrase_prompt | llm | StrOutputParser()
qa_chain = qa_prompt | llm
t = time.perf_counter(); await multi_agent.ainvoke(users[0]); print(f'1명 : {time.perf_counter() - t:.1f}초')
t = time.perf_counter(); await asyncio.gather(*(multi_agent.ainvoke(u) for u in users[:3])); print(f'3명 gather : {time.perf_counter() - t:.1f}초')

## 정리 — 비동기 처리 방식으로의 변화 3단계 (교재 p.125~128)

| 단계 | 바꾸는 것 | 주의 |
|---|---|---|
| 1 노드 | `def` → `async def` | 그래프 구성(`add_node` · `add_edge` · `compile`)은 그대로 |
| 2 I/O | `invoke` → `await … ainvoke` (체인 · 검색기 · LLM) | `async def` 안에서 동기 `invoke` · `time.sleep` 을 쓰면 루프 전체가 멈춤 |
| 3 실행 | `await agent.ainvoke(inputs)` 를 `async def main()` 에 · `.py` 는 `asyncio.run(main())` | Jupyter 는 `await main()` · 비동기 노드 그래프에 `invoke` → `TypeError` |
| 효과 | 한 대화는 같은 시간 · **여러 요청을 `gather` 로 동시에** 처리할 때 빨라짐 | 필기 `main()` 은 입력 없는 `ainvoke()` + `except` 로 **무한 반복** → 고침 |